In [211]:
import pandas as pd
fp = "../data/sba_loans_prepared/foia-7a-fy2020-present-as-of-251231.csv"
df = pd.read_csv(fp, low_memory=False)

In [212]:
df.head()

,asofdate,program,l2locid,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,...,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
0,12/31/2025,7A,507814.0,Plaza Drive Investments LLC,36223 PLAZA DR,CATHEDRAL CITY,CA,92234,"VelocitySBA, LLC",NaN,...,CORPORATION,Existing or more than 2 years old,EXEMPT,NaN,NaN,0.0,0,3,Y,Y
1,12/31/2025,7A,33850.0,Green Mountain Corporation,19301 S Santa Fe Ave,COMPTON,CA,90221,Comerica Bank,983.0,...,CORPORATION,Existing or more than 2 years old,PIF,10/31/2022,NaN,0.0,1,0,Y,NaN
2,12/31/2025,7A,112407.0,Robert D. Thompson and Lilia A. Garcia,8605 Sovereign Row,Dallas,TX,75247,Enterprise Bank & Trust,27237.0,...,CORPORATION,Existing or more than 2 years old,PIF,3/31/2023,NaN,0.0,0,4,Y,Y
3,12/31/2025,7A,85712.0,FordPowerSolutions LLC,520 SWEETWATER BRIDGE CIR,DOUGLASVILLE,GA,30134,United Midwest Savings Bank National Association,32441.0,...,CORPORATION,"Startup, Loan Funds will Open Business",EXEMPT,NaN,NaN,0.0,0,8,Y,Y
4,12/31/2025,7A,317954.0,Link Rec of Minong Inc.,304 Business Highway 53,MINONG,WI,54859,"Newtek Small Business Finance, Inc.",NaN,...,CORPORATION,Existing or more than 2 years old,PIF,4/30/2024,NaN,0.0,0,12,Y,Y


In [213]:
# infer default (raw) dtypes by reading a sample of the CSV and store in df_raw_dtypes
sample_nrows = 10000
df_sample = pd.read_csv(fp, low_memory=False, nrows=sample_nrows)
df_raw_dtypes = df_sample.dtypes.to_frame(name='raw_dtype')
df_raw_dtypes['raw_dtype'] = df_raw_dtypes['raw_dtype'].astype(str)
df_raw_dtypes

,raw_dtype
asofdate,str
program,str
l2locid,float64
borrname,str
borrstreet,str
borrcity,str
borrstate,str
borrzip,int64
bankname,str
bankfdicnumber,float64


In [214]:
# Collect attributes of each type in a list

# Numeric columns (float/int)
numeric_cols_list = df.select_dtypes(include=['int', "float"]).columns.tolist()

# Categorical columns
categorical_cols_list = df.select_dtypes(include=['category']).columns.tolist()

# Datetime columns
datetime_cols_list = df.select_dtypes(include=['datetime']).columns.tolist()

# String/object columns
string_cols_list = df.select_dtypes(include=['str']).columns.tolist()

print(f"Numeric columns: {numeric_cols_list}")
print(f"Categorical columns: {categorical_cols_list}")
print(f"Datetime columns: {datetime_cols_list}")
print(f"String columns: {string_cols_list}")

Numeric columns: ['l2locid', 'borrzip', 'bankfdicnumber', 'bankncuanumber', 'bankzip', 'grossapproval', 'sbaguaranteedapproval', 'approvalfiscalyear', 'initialinterestrate', 'terminmonths', 'naicscode', 'congressionaldistrict', 'grosschargeoffamount', 'revolverstatus', 'jobssupported']
Categorical columns: []
Datetime columns: []
String columns: ['asofdate', 'program', 'borrname', 'borrstreet', 'borrcity', 'borrstate', 'bankname', 'bankstreet', 'bankcity', 'bankstate', 'approvaldate', 'firstdisbursementdate', 'processingmethod', 'subprogram', 'fixedorvariableinterestind', 'naicsdescription', 'franchisecode', 'franchisename', 'projectcounty', 'projectstate', 'sbadistrictoffice', 'businesstype', 'businessage', 'loanstatus', 'paidinfulldate', 'chargeoffdate', 'collateralind', 'soldsecmrktind']


In [215]:
df = df.drop(["l2locid"], axis=1)

In [216]:
to_str_columns = ['borrzip', 'bankfdicnumber', 'bankncuanumber', 'bankzip']
df[to_str_columns] = df[to_str_columns].astype(str)
string_cols_list = string_cols_list + to_str_columns

In [217]:
N = df.shape[0]
cols  = df.columns.tolist()
print(f"Number of records: {N}")
nan_counts = {col: df[col].isna().sum().item() for col in cols}
df_dq_summ = pd.DataFrame.from_dict(nan_counts, orient='index', columns=['nan_count']) 
df_dq_summ['total_count'] = N
df_dq_summ['nan_percentage'] = df_dq_summ['nan_count'] / N * 100
df_dq_summ['nan_percentage'] = df_dq_summ['nan_percentage'].round(2)
df_dq_summ.sort_values(by='nan_percentage', ascending=False).head(10)

Number of records: 357866


,nan_count,total_count,nan_percentage
chargeoffdate,353006,357866,98.64
bankncuanumber,347811,357866,97.19
franchisename,315706,357866,88.22
franchisecode,315557,357866,88.18
paidinfulldate,301049,357866,84.12
soldsecmrktind,251402,357866,70.25
firstdisbursementdate,65730,357866,18.37
bankfdicnumber,37964,357866,10.61
naicsdescription,26998,357866,7.54
bankstreet,498,357866,0.14


In [218]:
df.loanstatus.unique()

<ArrowStringArray>
['EXEMPT', 'PIF', 'CHGOFF', 'CANCLD', 'COMMIT']
Length: 5, dtype: str

In [219]:
sel_charged_off = df['loanstatus'] == 'CHGOFF'
df_charged_off = df[sel_charged_off]
N_charged_off = df_charged_off.shape[0]
print(f"Number of charged off loans: {N_charged_off}")
df_charged_off.head()

Number of charged off loans: 4865


,asofdate,program,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,bankncuanumber,...,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
5,12/31/2025,7A,Chaiya Thai Corporation,272 CLAREMONT BLVD,SAN FRANCISCO,CA,94127,"U.S. Bank, National Association",6548.0,NaN,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaN,1/30/2025,3162.07,0,2,Y,NaN
8,12/31/2025,7A,Flypixe LLC,34116 CHAGRIN BLVD Apt 9105,CHAGRIN FALLS,OH,44022,"JPMorgan Chase Bank, National Association",628.0,NaN,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaN,4/2/2025,42368.47,1,0,Y,NaN
86,12/31/2025,7A,FORTUNE IMPORT & EXPORT INC,311 Bay 10th street FLoor 2,Brooklyn,NY,11228,"TD Bank, National Association",18409.0,NaN,...,CORPORATION,Unanswered,CHGOFF,NaN,12/14/2023,22111.14,0,0,N,NaN
90,12/31/2025,7A,Caffey Transportation LLC,807 IDLEWOOD DR,BAYTOWN,TX,77520,Wells Fargo Bank National Association,3511.0,NaN,...,CORPORATION,NaN,CHGOFF,NaN,3/21/2024,4852.20,1,0,N,NaN
107,12/31/2025,7A,Richard French,1875 Diesel Drive #9,SACRAMENTO,CA,95838,Five Star Bank,35361.0,NaN,...,INDIVIDUAL,Existing or more than 2 years old,CHGOFF,NaN,11/21/2024,151797.12,0,6,Y,Y


In [220]:
exclude_loanstatus = ['EXEMPT', 'CANCLD', "COMMIT"]
sel_exclude = df['loanstatus'].isin(exclude_loanstatus)
df_inculde = df[~sel_exclude]
df_inculde.loanstatus.unique()

<ArrowStringArray>
['PIF', 'CHGOFF']
Length: 2, dtype: str

In [221]:
df_exclude = df[sel_exclude]
df_exclude.shape

(296185, 42)

In [222]:
pif_select = df["loanstatus"] == "PIF"
df_pif = df[pif_select]
N_pif = df_pif.shape[0]
print(f"Number of PIF loans: {N_pif}")
df_pif.shape

Number of PIF loans: 56816


(56816, 42)

In [223]:
imbalance_ratio = N_charged_off / N_pif
print(f"Imbalance ratio (CHGOFF / PIF): {imbalance_ratio:.4f}")

Imbalance ratio (CHGOFF / PIF): 0.0856


In [224]:
str_cols = [col for col in df.columns if df[col].dtype == 'object']
date_cols = [col for col in df.columns if "date" in col.lower()]
na_numeric_cols = ["congressionaldistrict", "initialinterestrate"]

In [225]:
date_cols = [col for col in df.columns if "date" in col.lower()]
df[date_cols]= df[date_cols].fillna(pd.Timestamp("1900-01-01"))
df[string_cols_list] = df[string_cols_list].fillna("NA")
df[na_numeric_cols] = df[na_numeric_cols].fillna(-1)

In [226]:
df_null_report = df.isna().sum()
df_null_report.columns = ["null_count"]
df_null_report.sort_values(ascending=False).head(10)

asofdate          0
program           0
borrname          0
borrstreet        0
borrcity          0
borrstate         0
borrzip           0
bankname          0
bankfdicnumber    0
bankncuanumber    0
dtype: int64

In [227]:
## attribute type assignment

cols = df_charged_off.columns.tolist()
for col in cols:
    if col == "loanamount":
        df[col] = pd.to_numeric(df[col], errors='coerce')
    elif "date" in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')
    else:
        df[col] = df[col].astype('str')
        df[col] = df[col].astype('category')


In [228]:
dtype_dict = df.dtypes

In [229]:

dtype_dict = dtype_dict.to_dict()

In [230]:
# Convert to appropriate dtypes, handling missing categories
df_include = df_inculde.astype(dtype_dict, errors='ignore')
df_charged_off = df_charged_off.astype(dtype_dict, errors='ignore')
df_pif = df_pif.astype(dtype_dict, errors='ignore')

/tmp/ipykernel_103346/2050627543.py:2: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df_include = df_inculde.astype(dtype_dict, errors='ignore')
/tmp/ipykernel_103346/2050627543.py:2: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df_include = df_inculde.astype(dtype_dict, errors='ignore')
/tmp/ipykernel_103346/2050627543.py:2: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df_include = df_inculde.astype(dtype_dict, errors='ignore')
/tmp/ipykernel_103346/2050627543.py:2: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is d

In [231]:
df_clean = pd.concat([df_include, df_charged_off, df_pif], ignore_index=True)
del df

In [232]:
df_clean

,asofdate,program,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,bankncuanumber,...,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
0,2025-12-31,7A,Green Mountain Corporation,19301 S Santa Fe Ave,COMPTON,CA,90221,Comerica Bank,983.0,NaN,...,CORPORATION,Existing or more than 2 years old,PIF,2022-10-31,NaT,NaN,NaN,NaN,Y,NaN
1,2025-12-31,7A,Robert D. Thompson and Lilia A. Garcia,8605 Sovereign Row,Dallas,TX,75247,Enterprise Bank & Trust,27237.0,NaN,...,CORPORATION,Existing or more than 2 years old,PIF,2023-03-31,NaT,NaN,NaN,NaN,Y,Y
2,2025-12-31,7A,Link Rec of Minong Inc.,304 Business Highway 53,MINONG,WI,54859,"Newtek Small Business Finance, Inc.",NaN,NaN,...,CORPORATION,Existing or more than 2 years old,PIF,2024-04-30,NaT,NaN,NaN,NaN,Y,Y
3,2025-12-31,7A,Chaiya Thai Corporation,272 CLAREMONT BLVD,SAN FRANCISCO,CA,94127,"U.S. Bank, National Association",6548.0,NaN,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaT,2025-01-30,NaN,NaN,NaN,Y,NaN
4,2025-12-31,7A,Flypixe LLC,34116 CHAGRIN BLVD Apt 9105,CHAGRIN FALLS,OH,44022,"JPMorgan Chase Bank, National Association",628.0,NaN,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaT,2025-04-02,NaN,NaN,NaN,Y,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123357,2025-12-31,7A,MBD LANDSCAPE INC,314 Clark St,North Andover,MA,1845,"TD Bank, National Association",18409.0,NaN,...,CORPORATION,Existing or more than 2 years old,PIF,2023-06-30,NaT,NaN,NaN,NaN,N,NaN
123358,2025-12-31,7A,Odds & Ends Events LLC,313 N Evans St,TECUMSEH,MI,49286,The Huntington National Bank,6560.0,NaN,...,CORPORATION,New Business or 2 years or less,PIF,2023-10-31,NaT,NaN,NaN,NaN,Y,NaN
123359,2025-12-31,7A,Mcgar7 Transport LLC,7706 shepherdsville rd,ELIZABETHTOWN,KY,42701,Wilson & Muir Bank & Trust Company,17040.0,NaN,...,CORPORATION,"Startup, Loan Funds will Open Business",PIF,2025-07-31,NaT,NaN,NaN,NaN,Y,NaN
123360,2025-12-31,7A,Bodyworx Physical Therapy PLLC,"8809 S. Sooner Rd, Ste E",Oklahoma City,OK,73135,First Security Bank and Trust Company,17001.0,NaN,...,CORPORATION,Existing or more than 2 years old,PIF,2022-07-31,NaT,NaN,NaN,NaN,Y,NaN


In [233]:
cat_cols = df_clean.select_dtypes(include='category').columns.tolist()

In [234]:
sel_pif = df_clean["loanstatus"] == "PIF"
df_pif = df_clean[sel_pif]
N_pif = df_pif.shape[0]
sel_charged_off = df_clean['loanstatus'] == 'CHGOFF'
df_charged_off = df_clean[sel_charged_off]
N_charged_off = df_charged_off.shape[0]
imbalance_ratio = N_charged_off / N_pif
print(f"Imbalance ratio (CHGOFF / PIF) after cleaning: {imbalance_ratio:.4f}")

Imbalance ratio (CHGOFF / PIF) after cleaning: 0.0856


In [235]:
cols = df_clean.columns.tolist()
cols.remove("l2locid")
df_clean = df_clean[cols]

ValueError: list.remove(x): x not in list

In [ ]:
cat_cols = df_clean.select_dtypes(include='category').columns.tolist()

In [ ]:
bank_geo_cols = [ "bankcity", "bankstate", "bankstreet", "bankzip"]
borr_geo_cols = [ "borrstreet", "borrcity", "borrstate", "borrzip"]
numeric_cols = [" initialinterestrate", "terminmonths", "grossapproval","sbaguaranteedapproval", "businessage", "grosschargeoffamount"]
cat_cols = [col for col in cat_cols if col not in bank_geo_cols + borr_geo_cols + numeric_cols]

In [ ]:
df_clean["loanstatus"].value_counts()

loanstatus
PIF       113632
CHGOFF      9730
CANCLD         0
COMMIT         0
EXEMPT         0
Name: count, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data with default test size of 0.33
df_train, df_test = train_test_split(df_clean, test_size=0.33, random_state=42)

# Save to parquet files in the data directory
df_train.to_parquet("../data/df_clean_sba_train.parquet", index=False)
df_test.to_parquet("../data/df_clean_sba_test.parquet", index=False)

print(f"Training set size: {df_train.shape[0]}")
print(f"Test set size: {df_test.shape[0]}")

Training set size: 82652
Test set size: 40710


In [ ]:
df_clean.borrzip.dtype

CategoricalDtype(categories=['10001', '10002', '10003', '10004', '10005', '10006',
                  '10007', '10009', '1001', '10010',
                  ...
                  '99827', '99829', '99833', '99835', '99840', '99901',
                  '99921', '99925', '99928', '99929'],
, ordered=False, categories_dtype=str)